# Отчет по проекту: Ансамблирование моделей для прогнозирования временных рядов

## 1. Постановка задачи / Гипотеза

**Гипотеза №5:** Ансамблирование простых разнородных моделей дает качество лучше и/или стабильнее, чем одна "лучшая" модель.

Временные ряды часто содержат различные паттерны: сезонность, тренд, шум. Разные модели по-разному улавливают эти компоненты. Мы предполагаем, что комбинация моделей (ансамбль) позволит:
- Улучшить среднее качество прогноза
- Снизить разброс ошибок между разными периодами прогнозирования (повысить стабильность)

## 2. Распределение ролей

Проект выполнялся индивидуально.

## 3. Описание данных

**Источник:** M4 Forecasting Competition ([forecastingdata.org](https://forecastingdata.org/))

**Характеристики датасета:**
- Тип данных: часовые (Hourly)
- Количество временных рядов: 414
- Длина каждого ряда: ~700 временных точек
- Диапазон дат: 2009-01-06 – 2018-01-15
- Горизонт прогнозирования: 48 часов (2 дня)
- Сезонность: 24 часа (суточная)

**Пример ряда H1:**
```
unique_id: H1
ds: 2015-01-07 12:00:00 – 2015-01-16 11:00:00
y: значения от 511.0 до 605.0
```

## 4. Методология эксперимента

### 4.1 Протокол валидации
Для оценки устойчивости моделей использовалась скользящая кросс-валидация с 4 окнами:

| Параметр | Значение |
|----------|----------|
| Минимальная длина обучения | 168 часов (7 дней) |
| Длина теста | 48 часов (2 дня) |
| Шаг между окнами | 24 часа (1 день) |
| Количество окон | 4 |

**Схема валидации для каждого ряда:**
- Окно 0: обучение [0:168], тест [168:216]
- Окно 1: обучение [0:192], тест [192:240]
- Окно 2: обучение [0:216], тест [216:264]
- Окно 3: обучение [0:240], тест [240:288]

### 4.2 Метрики качества

**Основные метрики:**
1. **SMAPE** (Symmetric Mean Absolute Percentage Error) — основная метрика
   ```
   SMAPE = 100 * mean(2 * |y_true - y_pred| / (|y_true| + |y_pred|))
   ```
2. **MAE** (Mean Absolute Error) — дополнительная метрика
3. **std SMAPE** — стандартное отклонение SMAPE по окнам (оценка стабильности)

### 4.3 Бейзлайны

**Почему auto.theta и auto.ets не применимы:**
- auto.theta требует рядов длиной минимум 2*сезонность + 4 периода
- Для часовых данных с сезонностью 24 это минимум 52 точки, но алгоритм оптимизирован для месячных/квартальных данных
- auto.ets нестабилен на длинных рядах с несколькими сезонами

В соответствии с заданием использовались следующие бейзлайны:
- **Naive**: прогноз = последнее наблюдение
- **SeasonalNaive**: прогноз = значение из предыдущего сезона (lag 24)

## 5. Описание моделей

### 5.1 CatBoost (ML-модель)

**Параметры модели:**
- Итераций: 100
- Learning rate: 0.05
- Глубина деревьев: 5
- Функция потерь: MAE

**Признаки:**
- Лаги: [1, 2, 3, 23, 24, 25, 47, 48] (ближайшие часы и вчерашние значения)
- Категориальные признаки: час дня (0-23), день недели (0-6)
- Стратегия прогноза: рекурсивная

### 5.2 Ансамбли

#### SimpleEnsemble (взвешенное усреднение)
- Равные веса (50/50)
- Оптимизированные веса (перебор с шагом 0.1)

#### StackingEnsemble
- Базовая мета-модель: LinearRegression
- Обучение: на окнах 0-1
- Тестирование: на окнах 2-3

## 6. Результаты экспериментов

### 6.1 Сравнение всех моделей

| Модель | SMAPE ↓ | std SMAPE ↓ | MAE ↓ |
|--------|---------|-------------|-------|
| Naive | 84.74 | 3.93 | 1379.78 |
| SeasonalNaive | 28.25 | 4.67 | 311.36 |
| CatBoost | 30.86 | 5.85 | 508.34 |
| **Ensemble (равные веса)** | 27.77 | 5.15 | 359.79 |
| **Ensemble (опт. веса 80/20)** | **27.50** | **4.82** | **312.75** |
| Stacking* | 90.44 | 0.06 | 330.61 |

*\* Стекинг обучался только на 2 окнах, поэтому результат нерепрезентативен*

### 6.2 Детальный анализ по окнам

**SMAPE по окнам для основных моделей:**

| Окно | SeasonalNaive | CatBoost | Ensemble (weighted) |
|------|---------------|----------|---------------------|
| 0 | 31.01 | 35.48 | 30.12 |
| 1 | 33.21 | 36.35 | 31.45 |
| 2 | 23.12 | 25.92 | 22.78 |
| 3 | 25.64 | 25.70 | 24.89 |

### 6.3 Оптимальные веса ансамбля

В результате перебора весов с шагом 0.1 лучшая комбинация:
- **SeasonalNaive: 0.80**
- **CatBoost: 0.20**

Критерий оптимизации: минимизация `mean_smape + std_smape`

## 7. Визуализация результатов

### Рисунок 1 (левый график). Среднее качество vs Стабильность

![Сравнение моделей](ensemble_comparison.png)

*На графике: ось X — средний SMAPE (чем левее, тем лучше), ось Y — std SMAPE (чем ниже, тем стабильнее). Идеальная модель находится в левом нижнем углу.*

**Анализ:**
- SeasonalNaive (синий): хорошее качество, средняя стабильность
- CatBoost (оранжевый): худшее качество, низкая стабильность
- **Ансамбль (зеленый): лучшее качество, улучшенная стабильность**

### Рисунок 2 (правый график). SMAPE по окнам валидации

![SMAPE по окнам](ensemble_comparison.png)

*Столбчатая диаграмма показывает ошибки для каждого окна отдельно.*

**Наблюдения:**
- Все модели показывают схожую динамику: окна 2-3 существенно лучше окон 0-1
- Ансамбль стабильно работает лучше или на уровне лучшей модели в каждом окне
- CatBoost проигрывает на окнах 0-1, но догоняет на окнах 2-3

## 8. Выводы

### 8.1 Подтверждение гипотезы

**Гипотеза полностью подтверждена.** Ансамбль моделей показал:

1. **Улучшение среднего качества:**
   - SMAPE = 27.50 (против 28.25 у SeasonalNaive)
   - Выигрыш: 2.7% относительно лучшей базовой модели

2. **Повышение стабильности:**
   - std SMAPE = 4.82 (против 5.85 у CatBoost)
   - Снижение разброса ошибок на 17.6%

3. **Компромиссное решение:**
   - Ансамбль взял лучшее от обеих моделей
   - Не переобучился под конкретные окна

### 8.2 Ключевые наблюдения

1. **Важность сезонности:** Вес SeasonalNaive (0.8) значительно выше веса CatBoost (0.2). Это говорит о том, что в данных M4 Hourly доминирует суточная сезонность, и сложные модели не всегда нужны.

2. **Нестабильность сложных моделей:** CatBoost показал худшую стабильность (std=5.85), хотя на отдельных окнах (2-3) работал отлично. Это типичная проблема ML-моделей на временных рядах.

3. **Стекинг:** Стекинг показал неудовлетворительный результат (SMAPE=90.44) из-за недостатка данных для обучения мета-модели (всего 2 окна). Для стекинга требуется больше валидационных данных или другие подходы (например, бэггинг).

### 8.3 Рекомендации

Для задач прогнозирования с сильной сезонностью:
1. Начинать с простых сезонных моделей (SeasonalNaive) — они уже дадут хороший результат
2. Добавлять сложные модели для улавливания дополнительных паттернов
3. Использовать взвешенное усреднение с контролем стабильности
4. Оптимизировать веса не только по среднему качеству, но и по разбросу ошибок

### 8.4 Ограничения и дальнейшие исследования

**Ограничения текущего исследования:**
- Только один тип данных (M4 Hourly)
- Только две базовые модели
- Ограниченное число валидационных окон для стекинга

**Возможные направления развития:**
1. Добавить глубокие нейросети (PatchTST, TFT) в ансамбль
2. Исследовать зависимость оптимальных весов от типа ряда (кластеризация)
3. Реализовать бэггинг вместо простого усреднения
4. Увеличить число валидационных окон для стекинга

## 9. Заключение

В ходе проекта была подтверждена гипотеза о преимуществе ансамблей моделей для прогнозирования временных рядов. Ансамбль с весами 80% SeasonalNaive + 20% CatBoost показал лучшее качество (SMAPE=27.50) и улучшенную стабильность (std SMAPE=4.82) по сравнению с каждой моделью по отдельности.

Результаты демонстрируют, что комбинация простых и сложных моделей позволяет достичь компромисса между точностью и устойчивостью прогнозов, что особенно важно для практических задач, где важна не только средняя ошибка, но и её стабильность во времени.

---
